# Lab 2 — Construction & Expansion de la KB

**Prérequis :** avoir lancé le Lab 1 avant

On va construire un graphe RDF à partir des entités extraites, les aligner avec Wikidata, puis expandre la KB avec des requêtes SPARQL pour avoir assez de triplets pour le KGE.

In [ ]:
import json, time, hashlib, re, os, random, warnings
from collections import Counter
from typing import List, Tuple, Dict

import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.parse import quote

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ENTITIES_OUTPUT = r"extracted_knowledge.csv"
RELATIONS_OUTPUT = r"extracted_knowledge_relations.csv"
print("imports ok")


In [ ]:
OCT  = Namespace("http://octopusbiology.lab/ontology/")
ENT  = Namespace("http://octopusbiology.lab/entity/")
WDT  = Namespace("http://www.wikidata.org/prop/direct/")
WD   = Namespace("http://www.wikidata.org/entity/")
PROV = Namespace("http://www.w3.org/ns/prov#")

LABEL_CLASS = {
    "PER":     OCT.Researcher,
    "ORG":     OCT.Institution,
    "LOC":     OCT.Location,
    "MISC":    OCT.Species,
    "SPECIES": OCT.Species,
}

def slugify(text: str) -> str:
    text = re.sub(r"[^\w\s-]", "", text.strip())
    return re.sub(r"[\s]+", "_", text)[:80]

def make_entity_uri(text: str) -> URIRef:
    return ENT[slugify(text)]

def make_predicate_uri(verb: str) -> URIRef:
    return OCT[re.sub(r"[^\w]", "_", verb.strip().lower())]

print("✓ Namespaces et helpers définis")

## Etape 1 : Construction de la KB privée (TBox + ABox)

In [ ]:
entities_df  = pd.read_csv(ENTITIES_OUTPUT)
relations_df = pd.read_csv(RELATIONS_OUTPUT)
print(f"Entités   : {len(entities_df):,}  |  Relations : {len(relations_df):,}")

g = Graph()
for ns, prefix in [("oct", OCT), ("ent", ENT), ("owl", OWL),
                   ("rdfs", RDFS), ("rdf", RDF), ("prov", PROV),
                   ("wdt", WDT), ("wd", WD)]:
    g.bind(ns, prefix)

for uri, name, comment in [
    (OCT.Researcher,  "Researcher",  "Un chercheur en céphalopodes"),
    (OCT.Institution, "Institution", "Université ou laboratoire"),
    (OCT.Location,    "Location",    "Zone géographique"),
    (OCT.Species,     "Species",     "Espèce de céphalopode"),
    (OCT.Behavior,    "Behavior",    "Comportement observable"),
    (OCT.BodyPart,    "BodyPart",    "Partie anatomique"),
    (OCT.Habitat,     "Habitat",     "Environnement naturel"),
    (OCT.Predator,    "Predator",    "Prédateur des céphalopodes"),
    (OCT.Prey,        "Prey",        "Proie des céphalopodes"),
]:
    g.add((uri, RDF.type,        OWL.Class))
    g.add((uri, RDFS.subClassOf, OWL.Thing))
    g.add((uri, RDFS.label,      Literal(name, lang="fr")))
    g.add((uri, RDFS.comment,    Literal(comment, lang="fr")))

for prop, domain, range_ in [
    (OCT.habitatOf, OCT.Species,     OCT.Habitat),
    (OCT.preyOf,    OCT.Prey,        OCT.Species),
    (OCT.studiedBy, OCT.Species,     OCT.Researcher),
    (OCT.locatedIn, OCT.Institution, OCT.Location),
]:
    g.add((prop, RDF.type,    OWL.ObjectProperty))
    g.add((prop, RDFS.domain, domain))
    g.add((prop, RDFS.range,  range_))

entity_uri_map = {}
for _, row in tqdm(entities_df.iterrows(), total=len(entities_df), desc="Ajout entités"):
    uri = make_entity_uri(row["entity"])
    cls = LABEL_CLASS.get(row["label"], OCT.Species)
    g.add((uri, RDF.type,   cls))
    g.add((uri, RDFS.label, Literal(str(row["entity"]), lang="fr")))
    if pd.notna(row.get("source_url", "")) and str(row.get("source_url","")).startswith("http"):
        url_clean = str(row["source_url"]).replace(" ", "_")
        g.add((uri, PROV.wasDerivedFrom, URIRef(url_clean)))
    entity_uri_map[str(row["entity"])] = uri

skipped = 0  # ← initialisation
for _, row in tqdm(relations_df.iterrows(), total=len(relations_df), desc="Ajout relations"):
    subj_text = str(row["subject"])
    obj_text  = str(row["object"])
    verb      = str(row["relation"])

    if subj_text not in entity_uri_map:
        skipped += 1
        continue

    pred_uri = make_predicate_uri(verb)
    subj_uri = entity_uri_map[subj_text]
    obj_node = entity_uri_map[obj_text] if obj_text in entity_uri_map else Literal(obj_text, lang="fr")

    g.add((subj_uri, pred_uri, obj_node))
    g.add((pred_uri, RDF.type,   OWL.ObjectProperty))
    g.add((pred_uri, RDFS.label, Literal(verb, lang="fr")))

print(f"Triplets KB initiale : {len(g):,}  (skipped: {skipped:,})")
g.serialize("kb_initial.ttl", format="turtle")
print("→ Saved: kb_initial.ttl")

## Etape 2 : Entity Linking avec Wikidata

In [ ]:
WIKIDATA_SEARCH_URL = "https://www.wikidata.org/w/api.php"
CONFIDENCE_THRESHOLD = 0.80
BATCH_SIZE = 10

def reconcile_batch(labels: list) -> dict:
    results = {}
    for label in labels:
        try:
            params = {
                "action": "wbsearchentities",
                "search": label,
                "language": "fr",
                "uselang": "fr",
                "type": "item",
                "limit": 1,
                "format": "json",
            }
            headers = {"User-Agent": "OctopusLabBot/1.0 (academic; student@lab.fr)"}
            resp = requests.get(WIKIDATA_SEARCH_URL, params=params,
                                headers=headers, timeout=20)
            resp.raise_for_status()
            hits = resp.json().get("search", [])
            if hits:
                best = hits[0]
                # wbsearchentities ne donne pas de score de confiance,
                # on attribue 1.0 si exact, 0.85 sinon
                exact = best.get("label", "").lower() == label.lower()
                score = 1.0 if exact else 0.85
                results[label] = {
                    "id": best["id"],
                    "name": best.get("label", label),
                    "score": score,
                    "match": exact,
                }
            else:
                results[label] = None
        except Exception as e:
            print(f"  [Reconcile ERROR] {e}")
            results[label] = None
        time.sleep(0.1)  # petit délai entre chaque requête
    return results

candidates = entities_df[
    entities_df["label"].isin({"MISC", "LOC", "ORG"})
].drop_duplicates(subset="entity").head(500)

print(f"Entités candidates : {len(candidates):,}")

alignment_records = []
entity_labels_list = candidates["entity"].tolist()

for start in tqdm(range(0, len(entity_labels_list), BATCH_SIZE), desc="Réconciliation"):
    batch = entity_labels_list[start:start + BATCH_SIZE]
    results = reconcile_batch(batch)
    time.sleep(0.3)
    for label, hit in results.items():
        if hit is None:
            alignment_records.append({"private_entity": label, "external_uri": None,
                                       "external_label": None, "confidence": 0.0, "linked": False})
            continue
        wikidata_uri = WD[hit["id"]]
        confidence   = hit["score"]
        linked       = confidence >= CONFIDENCE_THRESHOLD
        alignment_records.append({"private_entity": label, "external_uri": str(wikidata_uri),
                                   "external_label": hit["name"],
                                   "confidence": round(confidence, 3), "linked": linked})
        if linked and label in entity_uri_map:
            g.add((entity_uri_map[label], OWL.sameAs, wikidata_uri))

alignment_df = pd.DataFrame(alignment_records)
print(f"\n✓ Liées : {alignment_df['linked'].sum()}/{len(alignment_df)} entités")
alignment_df.to_csv("alignment_table.csv", index=False)
print("→ Saved: alignment_table.csv")

## Etape 3 : Alignement des Prédicats + Expansion Wikidata

In [ ]:
for old_file in ["kb_expanded.ttl", "kb_expanded.nt"]:
    if os.path.exists(old_file):
        os.remove(old_file)
        print(f"✗ Supprimé : {old_file}")
print("✓ Anciens fichiers KB supprimés — reconstruction depuis zéro")

MANUAL_ALIGNMENT = {
    "appartenir": ("P171", "parent taxon"),    "appartient": ("P171", "parent taxon"),
    "classer":    ("P31",  "instance of"),      "être":       ("P31",  "instance of"),
    "vivre":      ("P551", "residence"),         "habiter":    ("P551", "residence"),
    "trouver":    ("P131", "located in"),        "distribuer": ("P131", "located in"),
    "pêcher":     ("P17",  "country"),
    "manger":     ("P1034","main food source"),  "consommer":  ("P1034","main food source"),
    "chasser":    ("P1672","preys on"),           "capturer":   ("P1672","preys on"),
    "avoir":      ("P527", "has part"),           "posséder":   ("P527", "has part"),
    "reproduire": ("P2567","mating system"),     "pondre":     ("P2567","mating system"),
    "utiliser":   ("P366", "use"),               "produire":   ("P1056","product produced"),
    "étudier":    ("P921", "main subject"),      "décrire":    ("P61",  "discoverer"),
    "nommer":     ("P138", "named after"),       "découvrir":  ("P61",  "discoverer"),
}

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
sparql_wd = SPARQLWrapper(WIKIDATA_SPARQL)
sparql_wd.addCustomHttpHeader("User-Agent", "OctopusLabBot/2.0 (academic; student@lab.fr)")
sparql_wd.setReturnFormat(JSON)

def run_wikidata_sparql(query: str) -> list:
    sparql_wd.setQuery(query)
    try:
        return sparql_wd.query().convert()["results"]["bindings"]
    except Exception as e:
        print(f"  [SPARQL ERROR] {e}")
        return []

def row_to_triple(row: dict):
    try:
        return row["s"]["value"], row["p"]["value"], row["o"]["value"]
    except KeyError:
        return None

private_predicates = relations_df["relation"].value_counts().head(30).index.tolist()
pred_align_records = []
for pred in private_predicates:
    keyword = pred.strip().lower().split("_")[0]
    if keyword in MANUAL_ALIGNMENT:
        pid, plabel = MANUAL_ALIGNMENT[keyword]
        pred_align_records.append({"private_predicate": pred, "wikidata_id": pid,
                                    "wikidata_label": plabel, "method": "manual"})
        g.add((make_predicate_uri(pred), OWL.equivalentProperty, WDT[pid]))
    else:
        pred_align_records.append({"private_predicate": pred, "wikidata_id": None,
                                    "wikidata_label": None, "method": "not_found"})

pred_align_df = pd.DataFrame(pred_align_records)
pred_align_df.to_csv("predicate_alignment.csv", index=False)
print(f"Prédicats alignés : {pred_align_df['wikidata_id'].notna().sum()}/{len(pred_align_df)}")

aligned_df = alignment_df[alignment_df["linked"]].copy()
aligned_df["qid"] = aligned_df["external_uri"].apply(lambda x: x.split("/")[-1] if x else None)
qids = aligned_df["qid"].dropna().unique().tolist()

# ANCHOR_QIDS corrigés — les vrais QIDs biologiques
# ATTENTION: Les anciens étaient FAUX (Q1077=film, Q192882=Irak, Q76125=personne)
ANCHOR_QIDS = [
    "Q40152",    # Octopoda (ordre des pieuvres)
    "Q26899",    # Cephalopoda (classe)
    "Q131257",   # Octopus (genre)
    "Q190782",   # Octopus vulgaris (poulpe commun)
    "Q1266322",  # Enteroctopus dofleini (pieuvre géante)
    "Q1476486",  # Hapalochlaena (pieuvres à anneaux bleus)
    "Q611843",   # Tremoctopus
    "Q891709",   # Amphioctopus marginatus
    "Q843338",   # Eledone cirrhosa
    "Q25341",    # Mollusca (phylum)
    "Q3348989",  # Incirrina (sous-ordre)
    "Q25823",    # Gastropoda (pour contraste taxonomique)
    "Q9266",     # Océan Pacifique
    "Q97",       # Océan Atlantique
    "Q4918",     # Mer Méditerranée
]
all_qids = list(set(qids + ANCHOR_QIDS))
print(f"QIDs totaux : {len(all_qids)}")

# Prédicats biologiques à privilégier pour l'expansion
EXPANSION_PREDICATES = [
    "wdt:P31", "wdt:P279", "wdt:P171", "wdt:P225", "wdt:P105",
    "wdt:P141", "wdt:P183", "wdt:P131", "wdt:P17",  "wdt:P1034",
    "wdt:P1672", "wdt:P527", "wdt:P361", "wdt:P2567", "wdt:P551",
    "wdt:P703", "wdt:P1542", "wdt:P2283", "wdt:P1552", "wdt:P186",
]

expanded_triples = set()

# 1-hop : FILTRÉ — seulement les prédicats pertinents
print("Expansion 1-hop (filtrée)...")
for qid in tqdm(all_qids, desc="1-hop"):
    rows = run_wikidata_sparql(f"""
    SELECT ?s ?p ?o WHERE {{
      BIND(wd:{qid} AS ?s)
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(isIRI(?o))
    }} LIMIT 500""")
    for row in rows:
        t = row_to_triple(row)
        if t:
            expanded_triples.add(t)
    time.sleep(0.5)
print(f"Triplets 1-hop : {len(expanded_triples):,}")

# Expansion par prédicats ciblés
print("Expansion par prédicats biologiques...")
for pred in tqdm(EXPANSION_PREDICATES, desc="Pred-controlled"):
    qid_values = " ".join(f"wd:{q}" for q in all_qids[:150])
    rows = run_wikidata_sparql(f"""
    SELECT ?s ?p ?o WHERE {{
      VALUES ?s {{ {qid_values} }}
      BIND({pred} AS ?p)
      ?s {pred} ?o .
      FILTER(isIRI(?o))
    }} LIMIT 3000""")
    for row in rows:
        t = row_to_triple(row)
        if t:
            expanded_triples.add(t)
    time.sleep(1.0)

# 2-hop CIBLÉ : seulement les entités biologiques (taxons)
print("Expansion 2-hop (ciblée taxonomie)...")
# Identifier les QIDs qui sont des taxons (ont P171 ou P105)
taxon_qids = set()
for s, p, o in expanded_triples:
    if "/P171" in p or "/P105" in p:
        for uri in [s, o]:
            if "wikidata.org/entity/Q" in uri:
                taxon_qids.add(uri.split("/")[-1])

sample_2hop = random.sample(list(taxon_qids), min(300, len(taxon_qids)))
print(f"  Taxons identifiés pour 2-hop : {len(taxon_qids)}, échantillon : {len(sample_2hop)}")

for qid in tqdm(sample_2hop, desc="2-hop"):
    rows = run_wikidata_sparql(f"""
    SELECT ?s ?p ?o WHERE {{
      BIND(wd:{qid} AS ?s)
      ?s ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(isIRI(?o))
    }} LIMIT 200""")
    for row in rows:
        t = row_to_triple(row)
        if t:
            expanded_triples.add(t)
    time.sleep(0.5)
print(f"Total triplets expandés : {len(expanded_triples):,}")

# Fusion dans le graphe
for s_str, p_str, o_str in tqdm(expanded_triples, desc="Ajout au graphe"):
    s_uri = URIRef(s_str)
    p_uri = URIRef(p_str)
    o_node = URIRef(o_str) if o_str.startswith("http") else Literal(o_str)
    g.add((s_uri, p_uri, o_node))

REMOVE_PREDS = [
    # Identifiants externes
    "P18", "P154", "P856", "P2671", "P213", "P214", "P227", "P244",
    "P268", "P269", "P349", "P496", "P646", "P935", "P373", "P910",
    "P1566", "P2002", "P2003", "P2013", "P3417", "P7859", "P8168",
    "P3984", "P4000", "P1711",
    # Géopolitique / cinéma / linguistique (bruit de l'ancien expansion)
    "P150", "P2936", "P530", "P463", "P161", "P725", "P1365", "P421",
    "P190", "P1343", "P47", "P832", "P974", "P1424", "P6104",
    "P1792", "P1740", "P1791", "P1313", "P1411", "P1709",
    "P8371", "P7867", "P1419", "P1717", "P1712",
    "P175", "P178", "P1056", "P8411", "P162", "P674", "P840",
    "P364", "P4969", "P1434", "P272", "P915", "P2512",
    "P3138", "P3121", "P3141", "P3123", "P734", "P20",
]
removed = 0
for pid in REMOVE_PREDS:
    p_uri = URIRef(f"http://www.wikidata.org/prop/direct/{pid}")
    triples = list(g.triples((None, p_uri, None)))
    removed += len(triples)
    for t in triples:
        g.remove(t)
print(f"Triplets supprimés (bruit) : {removed:,}")

# Stats finales
n_triples   = len(g)
n_entities  = len(set(g.subjects()) | {o for o in set(g.objects()) if isinstance(o, URIRef)})
n_relations = len(set(g.predicates()))
print(f"\n{'='*50}")
print(f"  Triplets  : {n_triples:>10,}")
print(f"  Entités   : {n_entities:>10,}")
print(f"  Relations : {n_relations:>10,}")
print(f"{'='*50}")

g.serialize("kb_expanded.ttl", format="turtle")
g.serialize("kb_expanded.nt",  format="nt")
print("→ Saved: kb_expanded.ttl + kb_expanded.nt")
print("\n✓ Lab 2 terminé.")
